# Weekly Loss Report
**Reporting week:** Oct 19-25, 2026  |  **Source:** company app (Supabase)  |  **Scope:** Waste only (Breakage, Item Search, Repeat Trip)

In [ ]:
import sys
from datetime import timedelta
sys.path.insert(0, "../src")
from reusables import *
from supabase_pull import load_waste, count_jobs, split_baseline_events, pull
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.2f}".format)

# ---- Edit these each week -------------------------------------------------
WEEK_START  = "2026-10-19"      # Monday of the reporting week
WEEK_LABEL  = "Week of Oct 19-25"
BASE_LABEL  = "Baseline"
BASE_JOBS   = 36                # jobs in the period the baseline covers (hand count)
WEEK_JOBS   = None              # None = count from the app; or type a number to override
LOAD_START  = "2026-01-01"      # far enough back to include the baseline
# ---------------------------------------------------------------------------

week_start = pd.Timestamp(WEEK_START).date()
week_end   = week_start + timedelta(days=7)          # following Monday (not included)

# All waste events up to the end of the week (app timestamps are UTC, so pull one extra day and filter on local date)
df = load_waste(LOAD_START, end=str(week_end + timedelta(days=1)))
df = df[df["incident_date"] < week_end].copy()
df["kind"] = df["kind"].replace({"Lost Item": "Item Search"})

baseline, after_baseline = split_baseline_events(df)
this_week = after_baseline[after_baseline["incident_date"] >= week_start]

if WEEK_JOBS is None:                                 # jobs with labor logged in the app that week
    WEEK_JOBS = count_jobs(str(week_start), str(week_end - timedelta(days=1)))

checks = run_checks(df, {
    "missing date":   df["incident_date"].isna(),
    "zero cost":      df["total_cost"].le(0),
    "no job linked":  df["job_id"].isna() & df["kind"].isin(["Breakage", "Reclean", "Repeat Trip"]),
})
print(f"{len(df)} events loaded | baseline {len(baseline)} | {WEEK_LABEL}: {len(this_week)} events, {WEEK_JOBS} jobs")


def compare_weeks(baseline, current, base_jobs, cur_jobs, base_label, cur_label):
    """Baseline vs reporting week by kind, with a Total row and per-100-jobs rates."""
    b = cost_by(baseline).rename(columns={"events": "b_events", "total": "b_cost"})
    c = cost_by(current).rename(columns={"events": "c_events", "total": "c_cost"})
    t = b.join(c, how="outer").fillna(0)
    t.loc["Total"] = t.sum()
    t["b_p100"] = (t["b_cost"] / base_jobs * 100).round(2)
    t["c_p100"] = (t["c_cost"] / cur_jobs * 100).round(2)
    t["chg"] = (t["c_p100"] - t["b_p100"]).round(2)
    t["chg_pct"] = ((t["c_p100"] / t["b_p100"] - 1) * 100).round(0)
    t = t.replace([float("inf"), -float("inf")], float("nan"))
    t = t[["b_events", "b_cost", "b_p100", "c_events", "c_cost", "c_p100", "chg", "chg_pct"]]
    t.columns = pd.MultiIndex.from_tuples([
        (base_label, "Events"), (base_label, "Cost ($)"), (base_label, "Per 100 jobs ($)"),
        (cur_label, "Events"), (cur_label, "Cost ($)"), (cur_label, "Per 100 jobs ($)"),
        ("Change", "$ per 100 jobs"), ("Change", "%")])
    return t


table = compare_weeks(baseline, this_week, BASE_JOBS, WEEK_JOBS, BASE_LABEL, WEEK_LABEL)

## Summary

In [ ]:
tot = table.loc["Total"]
cur  = tot[(WEEK_LABEL, "Per 100 jobs ($)")]
base = tot[(BASE_LABEL, "Per 100 jobs ($)")]
direction = "lower" if cur < base else "higher"
worst = table.drop("Total")[(WEEK_LABEL, "Cost ($)")].idxmax()

print(f"- {WEEK_LABEL}: {int(tot[(WEEK_LABEL, 'Events')])} waste events costing "
      f"${tot[(WEEK_LABEL, 'Cost ($)')]:,.2f} across {WEEK_JOBS} jobs.")
print(f"- Cost per 100 jobs was ${cur:,.2f}, {abs(cur - base) / base * 100:.0f}% "
      f"{direction} than the baseline (${base:,.2f}).")
print(f"- Largest cost driver this week: {worst}.")

## Waste by type: baseline vs reporting week
Costs are totals; "per 100 jobs" adjusts for how busy each period was. A blank change means the type has no baseline entries.

In [ ]:
table

## Waste Event Details

In [ ]:
cols = ["incident_date", "event_no", "kind", "job", "job_type", "title", "hours", "other_cost", "total_cost"]
cols = [c for c in cols if c in this_week.columns]       # skips any column your data doesn't have

details = (this_week[cols]
           .sort_values("total_cost", ascending=False)
           .rename(columns={"incident_date": "Date", "event_no": "Event", "kind": "Type", "job": "Job",
                            "job_type": "Job type", "title": "What happened", "hours": "Hours",
                            "other_cost": "Other ($)", "total_cost": "Total ($)"}))

(details.style
    .hide(axis="index")
    .format({"Hours": "{:.1f}", "Other ($)": "${:,.2f}", "Total ($)": "${:,.2f}"}, na_rep="")
    .set_properties(subset=["What happened"], **{"white-space": "normal", "text-align": "left", "min-width": "200px"}))

## Charts

In [ ]:
cost_events_chart(cost_by(this_week), title=f"{WEEK_LABEL}: Waste by type")

In [ ]:
weekly_cost_chart(
    {BASE_LABEL: baseline, WEEK_LABEL: this_week},
    jobs={BASE_LABEL: BASE_JOBS, WEEK_LABEL: WEEK_JOBS},
    title="Waste cost per 100 jobs",
)

## Data notes
- Costs come from the company app as entered: labor at $21.00/hr, mileage at $0.70/mile.
- Jobs this week = jobs with labor logged in the app. The baseline's 36 jobs is a hand count, so the two are not counted the same way.
- The baseline (events W-2026-0001 to W-2026-0014) is a historical import. It has no cause, crew or loss date, so it cannot be broken down further.
- Repeat Trip has no baseline entries because it was not tracked historically.
- A few weeks against an imported baseline show direction, not a trend.

## Data checks

In [ ]:
checks